# 从零写一个简单的 Transformer（Decoder-only）
## 任务：序列反转（Sequence Reversal）

本 Notebook 基于提供的 `transformer_tutorial.py` 整理。建议环境：Python 3、PyTorch。

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

PyTorch version: 2.2.2
CUDA available: False


# Self-Attention: Mathematical Formulation

## 1. Input Representation

给定输入序列：

$$
X = [x_1, x_2, \ldots, x_T]^\top
\in \mathbb{R}^{T \times d_{\text{model}}}
$$

其中：

* \(T\)：序列长度。
* \(d_{\text{model}}\)：每个 token 的 embedding 维度。
* \(x_i \in \mathbb{R}^{d_{\text{model}}}\)：第 \(i\) 个 token 的表示。

在 Transformer 中，输入表示通常结合 token embedding 和 positional encoding：

$$
x_i = e_i + p_i
$$

其中 \(e_i\) 是 token embedding，\(p_i\) 是位置编码。

Self-Attention 的目标是让每个 token 根据序列中其他相关 token 的信息，更新自己的表示。

## 2. Query, Key, and Value

首先，将输入 \(X\) 分别通过三个可学习的线性变换，得到 Query、Key 和 Value：

$$
Q = XW_Q,\qquad K = XW_K,\qquad V = XW_V
$$

其中：

$$
W_Q \in \mathbb{R}^{d_{\text{model}}\times d_k}
$$

$$
W_K \in \mathbb{R}^{d_{\text{model}}\times d_k}
$$

$$
W_V \in \mathbb{R}^{d_{\text{model}}\times d_v}
$$

因此：

$$
Q,K \in \mathbb{R}^{T\times d_k},
\qquad
V \in \mathbb{R}^{T\times d_v}
$$

对于第 \(i\) 个 token：

$$
q_i = x_iW_Q,\qquad
k_i = x_iW_K,\qquad
v_i = x_iW_V
$$

可以直观地理解为：

* **Query**：当前 token 想寻找什么信息。
* **Key**：每个 token 用于匹配的特征。
* **Value**：每个 token 实际提供的信息。

这三个矩阵都由模型训练学习得到，并不是人工指定的语义标签。

## 3. Attention Scores

通过 Query 和 Key 的点积，计算 token 之间的匹配分数：

$$
S = QK^\top
$$

由于：

$$
Q \in \mathbb{R}^{T\times d_k},
\quad
K^\top \in \mathbb{R}^{d_k\times T}
$$

所以：

$$
S \in \mathbb{R}^{T\times T}
$$

其中第 \(i\) 行、第 \(j\) 列元素为：

$$
s_{ij}=q_i k_j^\top
=\sum_{r=1}^{d_k}q_{ir}k_{jr}
$$

分数 \(s_{ij}\) 衡量第 \(i\) 个 token 的 Query 与第 \(j\) 个 token 的 Key 的匹配程度。

当 \(s_{ij}\) 较大时，模型通常会为 token \(j\) 分配较高的注意力权重；但最终权重还取决于同一行的其他分数。

## 4. Scaled Dot-Product Attention

如果直接对点积分数使用 Softmax，当 \(d_k\) 较大时，点积的幅度可能变大，导致 Softmax 的输出分布过于尖锐，梯度也可能变小。

因此，Transformer 使用缩放点积注意力：

$$
\boxed{
\operatorname{Attention}(Q,K,V)
=
\operatorname{softmax}
\left(
\frac{QK^\top}{\sqrt{d_k}}
\right)V
}
$$

缩放因子为：

$$
\frac{1}{\sqrt{d_k}}
$$

如果各维度的 Query 和 Key 分量独立、均值为零、方差为一，那么点积的方差近似为：

$$
\operatorname{Var}(q_i k_j^\top)=d_k
$$

除以 \(\sqrt{d_k}\) 后，方差近似恢复为：

$$
\operatorname{Var}
\left(
\frac{q_i k_j^\top}{\sqrt{d_k}}
\right)
\approx 1
$$

这有助于稳定 Softmax 的输入尺度。

## 5. Softmax and Attention Weights

定义缩放后的分数：

$$
Z=\frac{QK^\top}{\sqrt{d_k}}
$$

然后沿每一行应用 Softmax：

$$
A=\operatorname{softmax}(Z)
$$

逐元素表示为：

$$
\alpha_{ij}
=
\frac{\exp(z_{ij})}
{\sum_{k=1}^{T}\exp(z_{ik})}
$$

对于每个 Query \(i\)，注意力权重满足：

$$
\alpha_{ij}\geq 0,
\qquad
\sum_{j=1}^{T}\alpha_{ij}=1
$$

因此，每一行的注意力权重构成一个概率分布。

最终输出为：

$$
H=AV
$$

也就是说，第 \(i\) 个 token 的新表示是所有 Value 向量的加权和：

$$
h_i=\sum_{j=1}^{T}\alpha_{ij}v_j
$$

Self-Attention 可以通过这些权重，让每个 token 聚合其他 token 的信息。

## 6. Causal Mask in Decoder-Only Transformers

在 Decoder-only Transformer 中，预测当前位置的 token 时，不能访问未来位置的信息。因此需要使用因果掩码（Causal Mask）。

定义掩码矩阵：

$$
M_{ij}=
\begin{cases}
0, & j\leq i\\
-\infty, & j>i
\end{cases}
$$

将掩码加入 Attention Scores：

$$
Z=\frac{QK^\top}{\sqrt{d_k}}+M
$$

然后计算：

$$
A=\operatorname{softmax}(Z)
$$

因为：

$$
\exp(-\infty)=0
$$

所以对于 \(j>i\)：

$$
\alpha_{ij}=0
$$

这保证了第 \(i\) 个位置只能关注当前位置及之前的位置。

例如，当序列长度 \(T=4\) 时，掩码对应的可见性结构为：

$$
\begin{bmatrix}
1&0&0&0\\
1&1&0&0\\
1&1&1&0\\
1&1&1&1
\end{bmatrix}
$$

这里的 \(1\) 表示允许关注，\(0\) 表示禁止关注。

需要注意，掩码限制的是注意力的信息访问路径，并不意味着模型只能学习相邻 token 之间的关系。

## 7. Multi-Head Self-Attention

单个 Attention Head 只使用一组 Query、Key 和 Value 投影。Multi-Head Attention 使用多组独立的投影，使模型能够在不同的表示子空间中学习不同的关联模式。

对于第 \(h\) 个 Head：

$$
\operatorname{head}_h
=
\operatorname{Attention}
\left(
XW_Q^{(h)},
XW_K^{(h)},
XW_V^{(h)}
\right)
$$

如果共有 \(H\) 个 Heads，则先将各个输出拼接：

$$
\operatorname{Concat}
(\operatorname{head}_1,\ldots,\operatorname{head}_H)
$$

再通过输出投影矩阵：

$$
\boxed{
\operatorname{MultiHead}(X)
=
\operatorname{Concat}
(\operatorname{head}_1,\ldots,\operatorname{head}_H)W_O
}
$$

其中：

$$
W_O\in\mathbb{R}^{Hd_v\times d_{\text{model}}}
$$

常见的设计是让每个 Head 的维度为：

$$
d_k=d_v=\frac{d_{\text{model}}}{H}
$$

这样拼接后的维度为：

$$
Hd_v=d_{\text{model}}
$$

## 8. Computational Complexity

对于序列长度 \(T\) 和隐藏维度 \(d_{\text{model}}\)，Self-Attention 中计算 \(QK^\top\) 的主要复杂度为：

$$
O(T^2d_k)
$$

计算 \(AV\) 的复杂度为：

$$
O(T^2d_v)
$$

因此，忽略投影层以及常数因子后，Attention 的主要计算复杂度为：

$$
O(T^2d_{\text{model}})
$$

注意力分数矩阵的空间复杂度为：

$$
O(T^2)
$$

这意味着序列长度增加时，标准 Self-Attention 的计算和存储成本会近似按序列长度的平方增长。

## 9. Summary

Scaled Dot-Product Self-Attention 的完整计算过程可以概括为：

$$
\begin{aligned}
Q &= XW_Q\\
K &= XW_K\\
V &= XW_V\\
S &= \frac{QK^\top}{\sqrt{d_k}}\\
S_{\text{masked}} &= S+M\\
A &= \operatorname{softmax}(S_{\text{masked}})\\
H &= AV
\end{aligned}
$$

对于不需要因果约束的普通 Self-Attention，可以省略掩码 \(M\)。

对于 Decoder-only Transformer，因果掩码是自回归生成的重要组成部分。它使模型在训练时能够并行计算各个位置的表示，同时禁止每个位置利用未来 token 的信息。


## 1. Self-Attention（多头自注意力）

In [2]:
class SelfAttention(nn.Module):
    def __init__(self, dim, num_heads):
        super().__init__()
        assert dim % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5
        self.qkv = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)

    def forward(self, x, mask=None):
        B, T, D = x.shape
        qkv = self.qkv(x).reshape(B, T, 3, self.num_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)  # (3, B, H, T, d)
        q, k, v = qkv[0], qkv[1], qkv[2]

        att = (q @ k.transpose(-2, -1)) * self.scale  # (B, H, T, T)
        if mask is not None:
            att = att.masked_fill(mask == 0, float('-inf'))
        att = att.softmax(dim=-1)
        out = (att @ v).transpose(1, 2).reshape(B, T, D)
        return self.proj(out)

# Transformer Block（Transformer 模块）

Transformer Block 是 Transformer 模型的基本组成单元。它主要由两个部分组成：

1. **多头自注意力机制（Multi-Head Self-Attention, MHA）**
2. **前馈神经网络（Feed-Forward Network, FFN）**

每个部分都包含残差连接（Residual Connection）和层归一化（Layer Normalization）。

## 1. 输入表示

假设输入序列经过 Token Embedding 和 Positional Embedding 后，得到：

$$
X \in \mathbb{R}^{B \times T \times d_{\text{model}}}
$$

其中：

* \(B\)：批次大小（Batch Size）
* \(T\)：序列长度（Sequence Length）
* \(d_{\text{model}}\)：隐藏层维度（Hidden Dimension）

每个 Token 都由一个 \(d_{\text{model}}\) 维的向量表示。

## 2. 多头自注意力机制（MHA）

自注意力机制允许每个 Token 根据当前序列中的其他 Token，计算与它们之间的关联程度。

首先，通过线性变换得到 Query、Key 和 Value：

$$
Q=XW_Q,\qquad K=XW_K,\qquad V=XW_V
$$

然后计算注意力分数：

$$
S=\frac{QK^\top}{\sqrt{d_k}}
$$

对于 Decoder-only Transformer，需要使用因果掩码（Causal Mask），防止当前位置关注未来的 Token：

$$
A=\operatorname{softmax}
\left(
\frac{QK^\top}{\sqrt{d_k}}+M
\right)
$$

其中，\(M\) 是因果掩码：

* 允许关注的位置：\(M_{ij}=0\)
* 未来的位置：\(M_{ij}=-\infty\)

最后，计算注意力输出：

$$
\operatorname{Attention}(Q,K,V)=AV
$$

### 多头注意力（Multi-Head Attention）

多个注意力头可以分别学习不同类型的 Token 关系。

每个注意力头独立计算：

$$
\operatorname{head}_r=
\operatorname{Attention}(Q_r,K_r,V_r)
$$

然后将所有注意力头的输出拼接，并通过输出投影矩阵：

$$
\operatorname{MHA}(X)=
\operatorname{Concat}
(\operatorname{head}_1,\ldots,\operatorname{head}_H)W_O
$$

如果隐藏层维度为 \(8\)，并且有 \(4\) 个注意力头，那么每个头通常输出 \(2\) 维向量。拼接后重新得到 \(8\) 维向量。

## 3. 残差连接与层归一化

将多头注意力的输出与原始输入相加，再进行层归一化：

$$
H=\operatorname{LayerNorm}
\left(X+\operatorname{MHA}(X)\right)
$$

其中：

* **残差连接**：保留原始输入信息，有助于梯度传播。
* **层归一化**：稳定网络中的数值分布，使训练更稳定。

## 4. 前馈神经网络（FFN）

每个 Token 的隐藏表示都会独立通过同一个前馈神经网络。

典型的 FFN 由两个线性层和一个非线性激活函数组成：

$$
\operatorname{FFN}(h)=
W_2\sigma(W_1h+b_1)+b_2
$$

其中：

* \(W_1\)：将隐藏层维度扩展到更大的中间维度。
* \(\sigma\)：非线性激活函数，例如 GELU。
* \(W_2\)：将中间维度映射回隐藏层维度。

例如：

$$
8\rightarrow32\rightarrow8
$$

FFN 不会直接混合不同 Token 的信息，而是对每个 Token 的表示分别进行特征变换。

## 5. 第二次残差连接与层归一化

将 FFN 的输出与输入相加，再进行层归一化：

$$
Y=\operatorname{LayerNorm}
\left(H+\operatorname{FFN}(H)\right)
$$

因此，一个 Transformer Block 的完整计算过程为：

$$
H=\operatorname{LayerNorm}
\left(X+\operatorname{MHA}(X)\right)
$$

$$
Y=\operatorname{LayerNorm}
\left(H+\operatorname{FFN}(H)\right)
$$

输出 \(Y\) 的形状仍然是：

$$
Y\in\mathbb{R}^{B\times T\times d_{\text{model}}}
$$

## 6. 多个 Transformer Block 的堆叠

Transformer 模型通常会堆叠多个 Transformer Block：

$$
X^{(1)}=\operatorname{Block}_1(X^{(0)})
$$

$$
X^{(2)}=\operatorname{Block}_2(X^{(1)})
$$

$$
\cdots
$$

$$
X^{(L)}=\operatorname{Block}_L(X^{(L-1)})
$$

其中，\(L\) 表示 Transformer Block 的数量。

随着网络深度增加，每个 Token 的隐藏表示能够逐步整合更丰富的上下文信息。

## 7. 从 Transformer Block 到下一个 Token 的预测

在 Decoder-only Transformer 中，最后一个 Block 的输出通常会经过最终层归一化，再通过语言模型输出层（LM Head）：

$$
\operatorname{Logits}=X^{(L)}W_{\text{vocab}}
$$

Logits 的形状为：

$$
\mathbb{R}^{B\times T\times |\mathcal{V}|}
$$

其中，\(|\mathcal{V}|\) 是词表大小。

对每个位置的 Logits 进行 Softmax，可以得到词表中各个 Token 的概率：

$$
P(x_{t+1}\mid x_1,\ldots,x_t)
=
\operatorname{softmax}(\operatorname{Logits}_t)
$$

模型据此预测下一个 Token，并可以逐步生成完整序列。

## 总结

一个 Transformer Block 的核心流程是：

**输入表示 → 多头自注意力 → 残差连接与层归一化 → 前馈神经网络 → 残差连接与层归一化 → 输出表示**

需要注意：以上公式采用的是 **Post-LayerNorm** 结构。现代 GPT 类模型通常采用 **Pre-LayerNorm** 或其变体，具体结构会有所不同。


## 2. Transformer Block

In [3]:
class Block(nn.Module):
    def __init__(self, dim, num_heads, mlp_ratio=4.0, dropout=0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = SelfAttention(dim, num_heads)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, int(dim * mlp_ratio)),
            nn.GELU(),
            nn.Linear(int(dim * mlp_ratio), dim),
            nn.Dropout(dropout),
        )

    def forward(self, x, mask=None):
        x = x + self.attn(self.norm1(x), mask)
        x = x + self.mlp(self.norm2(x))
        return x

# TinyTransformer：从零理解一个小型 Transformer

## 1. 什么是 TinyTransformer？

TinyTransformer 是一个小型 Transformer 模型，适合用来学习 Attention、Embedding、Transformer Block 和自回归生成等核心概念。

它可以用于学习两类任务：

* **自回归语言建模（Autoregressive Language Modeling）**：根据前面的 token 预测下一个 token。
* **序列到序列任务（Sequence-to-Sequence）**：根据输入序列生成目标序列，例如把输入序列反转。

需要注意：TinyTransformer 并不是一种固定的标准架构。具体能力取决于模型结构、训练目标和数据组织方式。

## 2. 模型的主要组成部分

一个典型的 Decoder-only TinyTransformer 包含以下组件：

1. Token Embedding
2. Positional Embedding
3. Transformer Blocks

   * Causal Self-Attention
   * Feed-Forward Network（FFN）
   * Residual Connection
   * Layer Normalization
4. Final LayerNorm
5. Language Model Head（LM Head）

整体结构：

```text
Input Token IDs
       |
       v
Token Embedding + Positional Embedding
       |
       v
Transformer Block 1
       |
       v
Transformer Block 2
       |
      ...
       |
       v
Final LayerNorm
       |
       v
LM Head
       |
       v
Logits: (B, T, vocab_size)
```

其中：

* `B`：Batch Size，即一次处理多少条序列。
* `T`：Sequence Length，即每条序列包含多少个 token。
* `dim`：隐藏向量维度。
* `vocab_size`：词表大小。

## 3. Token Embedding

```python
self.tok_emb = nn.Embedding(vocab_size, dim)
```

Token Embedding 将每个整数 token ID 映射成一个可训练的向量。

例如：

```text
Token IDs:          (B, T)
Token Embeddings:   (B, T, dim)
```

假设：

```python
B = 32
T = 6
dim = 64
```

则 Embedding 输出形状为：

```python
(32, 6, 64)
```

每个 token 都由一个 64 维向量表示。

## 4. Positional Embedding

```python
self.pos_emb = nn.Embedding(max_len, dim)
```

Self-Attention 本身不直接提供 token 的绝对位置，因此模型需要额外的位置表示。

在这个实现中，位置向量与 Token Embedding 相加：

```python
pos = torch.arange(T, device=x.device).unsqueeze(0)

x = self.tok_emb(x) + self.pos_emb(pos)
```

两者的形状为：

```text
Token Embedding:      (B, T, dim)
Position Embedding:   (1, T, dim)
                       |
                       v
Output:               (B, T, dim)
```

PyTorch 会利用 broadcasting，将位置向量应用到 batch 中的每条序列。

## 5. Self-Attention 与 Causal Mask

Self-Attention 让每个 token 根据相关性，从其他可见 token 中获取信息。

在 GPT 类自回归模型中，需要使用 Causal Mask，防止当前位置看到未来 token。

```python
mask = torch.tril(
    torch.ones(T, T, device=x.device)
).unsqueeze(0).unsqueeze(0)
```

当 `T = 4` 时，下三角矩阵为：

```text
1 0 0 0
1 1 0 0
1 1 1 0
1 1 1 1
```

其中：

* `1`：允许关注。
* `0`：禁止关注。

Attention 计算中必须真正应用这个 mask，例如将被屏蔽位置的 attention logits 设为负无穷，再执行 Softmax。

Causal Mask 与 Padding Mask 不同：

* Causal Mask：防止看到未来 token。
* Padding Mask：防止模型关注为了对齐序列长度而添加的 PAD token。

## 6. Multi-Head Attention

Multi-Head Attention 将注意力计算分成多个头，让模型学习不同的 token 关系。

假设：

```python
dim = 64
num_heads = 4
```

每个头的维度为：

```python
head_dim = dim // num_heads
# 16
```

通常，输入经过 Q、K、V 投影后，会被拆分成多个头。各个头独立计算 Attention，之后拼接并通过输出投影，恢复到原来的隐藏维度。

典型的形状变化：

```text
Input:                 (B, T, 64)
Q, K, V:               (B, 4, T, 16)
Attention Output:      (B, 4, T, 16)
Concatenated Output:   (B, T, 64)
```

最终输出仍然是 `(B, T, dim)`。

## 7. Feed-Forward Network（FFN）

FFN 对每个 token 的隐藏表示进行非线性变换。

典型结构：

```python
self.ffn = nn.Sequential(
    nn.Linear(dim, hidden_dim),
    nn.GELU(),
    nn.Linear(hidden_dim, dim)
)
```

FFN 的作用与 Attention 不同：

* Attention：在 token 之间交换信息。
* FFN：对每个 token 已经融合上下文信息的表示进行进一步加工。

同一个 FFN 的参数会应用于序列的每个位置。

FFN 通常在 Multi-Head Attention 完成多头拼接和输出投影之后执行，而不是分别对每个 Attention Head 独立执行。

## 8. Residual Connection 与 LayerNorm

Residual Connection 保留子层输入，并提供直接的梯度传播路径。

一种 Post-LN 写法：

```python
x = self.norm1(x + self.attn(x, mask))
x = self.norm2(x + self.ffn(x))
```

一种 Pre-LN 写法：

```python
x = x + self.attn(self.norm1(x), mask)
x = x + self.ffn(self.norm2(x))
```

这两种写法的归一化顺序不同。现代 GPT 类模型通常采用 Pre-LN 或相关变体。

具体实现应以实际 `Block` 类为准。

## 9. TinyTransformer 的初始化

```python
class TinyTransformer(nn.Module):
    def __init__(
        self,
        vocab_size,
        dim=64,
        depth=2,
        num_heads=4,
        max_len=32
    ):
        super().__init__()

        self.tok_emb = nn.Embedding(vocab_size, dim)
        self.pos_emb = nn.Embedding(max_len, dim)

        self.blocks = nn.ModuleList([
            Block(dim, num_heads)
            for _ in range(depth)
        ])

        self.norm = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, vocab_size)

        self.max_len = max_len
```

参数含义：

| 参数           | 含义                           |
| ------------ | ---------------------------- |
| `vocab_size` | 词表大小                         |
| `dim`        | 隐藏维度                         |
| `depth`      | Transformer Block 数量         |
| `num_heads`  | 每个 Block 的 Attention Head 数量 |
| `max_len`    | 支持的最大序列长度                    |

注意：`depth` 是 Block 数量，`num_heads` 是每个 Block 中的 Attention Head 数量，两者不是同一个概念。

## 10. Forward：从输入到输出

```python
def forward(self, x):
    B, T = x.shape

    pos = torch.arange(T, device=x.device).unsqueeze(0)

    x = self.tok_emb(x) + self.pos_emb(pos)

    mask = torch.tril(
        torch.ones(T, T, device=x.device)
    ).unsqueeze(0).unsqueeze(0)

    for blk in self.blocks:
        x = blk(x, mask)

    x = self.norm(x)

    return self.head(x)
```

假设：

```python
B = 32
T = 6
dim = 64
vocab_size = 22
```

张量形状变化：

| 阶段                         | 输出形状          |
| -------------------------- | ------------- |
| 输入 token IDs               | `(32, 6)`     |
| Token + Position Embedding | `(32, 6, 64)` |
| 每个 Transformer Block       | `(32, 6, 64)` |
| Final LayerNorm            | `(32, 6, 64)` |
| LM Head                    | `(32, 6, 22)` |

最终输出是 logits，而不是概率。每个序列位置都有 22 个 token 的预测分数。

训练时通常直接将 logits 传给 `nn.CrossEntropyLoss`，不必提前执行 Softmax。

## 11. 训练任务：反转序列

可以使用合成数据，训练模型学习将输入序列反转。

```python
VOCAB = 22
BOS = 1

def make_batch(batch_size=32, length=6, device="cpu"):
    src = torch.randint(
        2, VOCAB,
        (batch_size, length),
        device=device
    )

    tgt = torch.cat(
        [
            torch.full(
                (batch_size, 1),
                BOS,
                dtype=torch.long,
                device=device
            ),
            src.flip(1)
        ],
        dim=1
    )

    return src, tgt
```

例如：

```text
src = [5, 8, 3, 12, 7, 4]
tgt = [1, 4, 7, 12, 3, 8, 5]
```

其中：

* `src`：原始输入。
* `src.flip(1)`：沿序列维度反转。
* `BOS`：目标序列的起始标记。
* `tgt`：模型应该生成的目标序列。

这个任务可以检验模型能否学会序列映射规律，但它不是自然语言理解任务。

## 12. 训练时的输入与目标

对于自回归模型，通常使用 Teacher Forcing：训练时把正确的前缀作为输入，让模型预测下一个 token。

目标序列：

```text
[BOS, 4, 7, 12, 3, 8, 5]
```

Decoder 输入：

```text
[BOS, 4, 7, 12, 3, 8]
```

训练标签：

```text
[4, 7, 12, 3, 8, 5]
```

代码示意：

```python
decoder_input = tgt[:, :-1]
labels = tgt[:, 1:]
```

模型需要学习：

```text
BOS                 -> 4
BOS, 4              -> 7
BOS, 4, 7           -> 12
BOS, 4, 7, 12       -> 3
...
```

**重要：** 上述移位方法适用于 Decoder 已经能获得源序列信息的架构。若使用 Encoder–Decoder Transformer，应将 `src` 编码后提供给 Decoder；若使用 Decoder-only Transformer，则必须另外设计输入拼接、分隔标记和 loss mask 等机制，让模型能够根据 `src` 生成反转序列。不能仅将 `src` 传入前面的 `TinyTransformer` 就自动完成条件反转任务。

## 13. 训练与生成的区别

**训练阶段：**

* 提供输入和正确目标。
* 计算 logits 与目标 token 之间的交叉熵损失。
* 通过反向传播更新模型参数。

**生成阶段：**

* 从 BOS 或指定的提示序列开始。
* 预测下一个 token。
* 选择或采样一个 token。
* 将它添加到序列中。
* 重复预测，直到达到停止条件。

## 14. 核心总结

TinyTransformer 的核心流程是：

```text
Token IDs
   ↓
Embedding
   ↓
Attention：token 之间交换信息
   ↓
FFN：加工 token 表示
   ↓
LayerNorm
   ↓
LM Head
   ↓
Next-token logits
```

对于自回归语言模型，核心目标是预测下一个 token；通过不断重复这个过程，可以生成更长的文本。

对于序列反转任务，模型学习的是从输入序列到反转序列的映射。架构、训练数据和 loss 的设计必须彼此匹配，才能让模型真正学会这个任务。


## 3. TinyTransformer

In [4]:
class TinyTransformer(nn.Module):
    def __init__(self, vocab_size, dim=64, depth=2, num_heads=4, max_len=32):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, dim)
        self.pos_emb = nn.Embedding(max_len, dim)
        self.blocks = nn.ModuleList([Block(dim, num_heads) for _ in range(depth)])
        self.norm = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, vocab_size)
        self.max_len = max_len

    def forward(self, x):
        B, T = x.shape
        pos = torch.arange(T, device=x.device).unsqueeze(0)
        x = self.tok_emb(x) + self.pos_emb(pos)

        # 因果 mask：位置 i 只能看到 <= i
        mask = torch.tril(torch.ones(T, T, device=x.device)).unsqueeze(0).unsqueeze(0)

        for blk in self.blocks:
            x = blk(x, mask)
        x = self.norm(x)
        return self.head(x)  # (B, T, vocab_size)

## 4. 玩具任务：序列反转

In [15]:
VOCAB = 22
BOS = 1

def make_batch(batch_size=32, length=6, device='cpu'):
    src = torch.randint(2, VOCAB, (batch_size, length), device=device)
    # 目标：BOS + 反转后的序列
    tgt = torch.cat(
        [torch.full((batch_size, 1), BOS, dtype=torch.long, device=device), src.flip(1)],
        dim=1,
    )
    return src, tgt

def make_batch2(batch_size=32, length=6, device='cpu'):
    # 随机生成 2~21 之间的 token ID
    src = torch.randint(2, VOCAB, (batch_size, length), device=device)

    # 循环减 1：2 -> 21，其他 token 都减 1
    shifted = 2 + torch.remainder(src - 3, VOCAB - 2)

    # 目标序列：[BOS, 循环减 1 后的序列]
    bos = torch.full(
        (batch_size, 1), BOS, dtype=torch.long, device=device
    )
    tgt = torch.cat([bos, shifted], dim=1)

    return src, tgt

# TinyTransformer：训练过程详解

## 1. 训练函数的整体流程

训练的目标是让 TinyTransformer 学会一个序列反转任务：

给定输入序列 `src`，模型学习生成带有 `BOS` 标记的反转序列 `tgt`。

例如：

```text
src   = [5, 8, 3, 12, 7, 4]
tgt   = [1, 4, 7, 12, 3, 8, 5]
```

其中 `1` 是 `BOS`（Beginning Of Sequence），表示目标序列的起始位置。

训练过程可以概括为：

```text
初始化模型
    ↓
生成一批训练数据
    ↓
构造模型输入 inp 和训练标签 label
    ↓
模型前向传播，计算 logits
    ↓
计算 Cross-Entropy Loss
    ↓
反向传播，计算梯度
    ↓
Optimizer 更新模型参数
    ↓
重复训练，直到完成所有 steps
```

## 2. 选择计算设备

```python
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', device)
```

这段代码自动选择计算设备：

* `cuda`：如果 PyTorch 检测到可用的 CUDA GPU，就使用 GPU。
* `cpu`：如果没有可用的 CUDA GPU，就使用 CPU。

打印出来的结果可能是：

```text
Using device: cuda
```

或者：

```text
Using device: cpu
```

注意：Mac 上的 Apple Silicon GPU 通常使用 PyTorch 的 `mps` 后端，而不是 CUDA。当前代码不会主动选择 `mps`，因此在没有 CUDA 的 Mac 上通常会使用 CPU。

## 3. 固定随机种子

```python
torch.manual_seed(0)
```

设置 PyTorch 的随机种子，帮助实验获得可复现的结果。

它会影响许多 PyTorch 随机操作，例如模型参数的初始化和随机张量生成。

不过，这不保证所有设备和所有操作都能完全确定性复现。

## 4. 初始化 TinyTransformer

```python
model = TinyTransformer(
    vocab_size=VOCAB,
    dim=64,
    depth=2,
    num_heads=4,
    max_len=16
).to(device)
```

这里创建一个 TinyTransformer 实例。

| 参数           |  值 | 含义                           |
| ------------ | -: | ---------------------------- |
| `vocab_size` | 22 | 词表大小                         |
| `dim`        | 64 | 每个 token 的隐藏向量维度             |
| `depth`      |  2 | Transformer Block 的数量        |
| `num_heads`  |  4 | 每个 Block 的 Attention Head 数量 |
| `max_len`    | 16 | 位置 Embedding 支持的最大长度         |

`.to(device)` 将模型参数移动到选定的设备上。

因为 `tgt` 长度为 7，而训练输入 `inp` 长度为 6，所以 `max_len=16` 足够容纳当前输入。

## 5. 创建优化器

```python
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4
)
```

优化器负责根据反向传播得到的梯度更新模型参数。

* `model.parameters()`：需要训练的模型参数，例如 Embedding、Attention、FFN 和 LM Head 的权重。
* `AdamW`：一种常用的梯度优化算法。
* `lr=3e-4`：学习率，即 `0.0003`，控制每次参数更新的步幅。

可以把它理解为：模型先通过训练发现自己哪里预测得不好，再通过优化器调整参数，让下一次预测更接近正确答案。

## 6. 训练循环

```python
for step in range(2001):
```

`range(2001)` 会产生从 `0` 到 `2000` 的整数，因此一共执行 **2001 个训练步骤**。

每个 step 都会生成一批新的随机数据，并完成一次参数更新。

这里的 step 不等于 epoch。由于数据是在每个 step 中随机生成的，这段代码并没有遍历一个固定的数据集。

## 7. 生成训练数据

```python
src, tgt = make_batch(64, 6, device)
```

调用之前定义的 `make_batch()`：

```python
def make_batch(batch_size=32, length=6, device='cpu'):
    src = torch.randint(
        2, VOCAB,
        (batch_size, length),
        device=device
    )

    tgt = torch.cat(
        [
            torch.full(
                (batch_size, 1),
                BOS,
                dtype=torch.long,
                device=device
            ),
            src.flip(1)
        ],
        dim=1
    )

    return src, tgt
```

这里指定：

* `batch_size=64`：一次训练 64 条序列。
* `length=6`：每条输入序列有 6 个 token。

因此：

```text
src.shape = (64, 6)
tgt.shape = (64, 7)
```

输入序列中的 token ID 来自 `[2, 21]`，而目标序列是 `BOS + 反转后的 src`。

注意：这份训练代码生成了 `src`，但后续并没有把 `src` 直接传入模型。这一点在第 12 节会进一步说明。

## 8. 构造 inp 和 label：Teacher Forcing

这是训练代码中最重要的部分之一。

```python
inp = tgt[:, :-1]
label = tgt[:, 1:]
```

原始目标序列：

```text
tgt = [BOS, 4, 7, 12, 3, 8, 5]
```

### 8.1 构造模型输入

```python
inp = tgt[:, :-1]
```

`:` 表示保留 batch 中的所有样本；`:-1` 表示保留序列中除最后一个 token 以外的所有 token。

结果：

```text
inp = [BOS, 4, 7, 12, 3, 8]
```

### 8.2 构造训练标签

```python
label = tgt[:, 1:]
```

`1:` 表示从序列的第二个 token 开始，一直取到末尾。

结果：

```text
label = [4, 7, 12, 3, 8, 5]
```

对应关系：

| 模型看到的 token           | 正确的预测目标 |
| --------------------- | ------- |
| `BOS`                 | `4`     |
| `BOS, 4`              | `7`     |
| `BOS, 4, 7`           | `12`    |
| `BOS, 4, 7, 12`       | `3`     |
| `BOS, 4, 7, 12, 3`    | `8`     |
| `BOS, 4, 7, 12, 3, 8` | `5`     |

这就是 **Teacher Forcing（教师强制）**：训练时直接使用正确的目标前缀，而不是使用模型自己上一步预测出来的 token。

这样可以在一次前向传播中，同时训练序列中的多个预测位置。

由于：

```text
tgt.shape   = (64, 7)
inp.shape   = (64, 6)
label.shape = (64, 6)
```

输入和标签在序列长度上对齐，每个输入位置都有一个对应的预测目标。

## 9. 前向传播：得到 logits

```python
logits = model(inp)
```

将 `inp` 输入 TinyTransformer。

模型内部会依次执行：

```text
Token Embedding + Positional Embedding
             ↓
Transformer Block 1
             ↓
Transformer Block 2
             ↓
Final LayerNorm
             ↓
LM Head
             ↓
Logits
```

因为：

```text
inp.shape = (64, 6)
```

并且：

```text
dim = 64
VOCAB = 22
```

所以输出形状为：

```text
logits.shape = (64, 6, 22)
```

其中：

* `64`：batch size。
* `6`：每条序列需要预测的位置数。
* `22`：每个位置对词表中 22 个 token 的预测分数。

Logits 是未经 Softmax 归一化的分数，不是概率。

例如，对某个位置，模型可能输出：

```text
Token 0: -1.2
Token 1:  0.5
Token 2:  2.1
...
Token 21: 0.3
```

分数越高，模型越倾向于预测对应的 token。

## 10. 计算 Cross-Entropy Loss

```python
loss = F.cross_entropy(
    logits.reshape(-1, VOCAB),
    label.reshape(-1)
)
```

Loss 用来衡量模型预测结果与正确答案之间的差距。

### 10.1 为什么要 reshape？

原始形状：

```text
logits: (64, 6, 22)
label:  (64, 6)
```

`F.cross_entropy()` 处理分类任务时，可以把每个预测位置当作一个独立的分类样本。

因此将 logits 转换为：

```python
logits.reshape(-1, VOCAB)
```

得到：

```text
(64 × 6, 22) = (384, 22)
```

每一行代表一个位置对 22 个 token 的预测分数。

同时：

```python
label.reshape(-1)
```

将标签转换为：

```text
(384,)
```

其中每个整数表示该位置的正确 token ID。

### 10.2 Loss 的意义

Cross-Entropy Loss 会惩罚模型给正确 token 分配过低概率的情况。

* Loss 较高：模型通常还没有学会正确的预测规律。
* Loss 下降：模型的预测总体上正在接近正确标签。
* Loss 接近 0：模型对训练样本中的正确 token 赋予了很高的概率。

但 Loss 下降并不保证模型一定能对所有未见过的序列正确反转，还需要实际测试生成结果。

## 11. 反向传播与参数更新

### 11.1 清除旧梯度

```python
optimizer.zero_grad()
```

PyTorch 默认会累积梯度，因此在当前 step 开始时清除之前的梯度，避免意外累加。

### 11.2 计算梯度

```python
loss.backward()
```

反向传播会根据 Loss 计算各个可训练参数的梯度。

这些梯度反映了参数变化对 Loss 的影响方向和大小。

### 11.3 更新参数

```python
optimizer.step()
```

AdamW 根据计算得到的梯度更新模型参数。

完整流程是：

```text
Forward
   ↓
计算 Loss
   ↓
zero_grad()
   ↓
backward()
   ↓
optimizer.step()
```

这里的 `zero_grad()` 通常放在 `backward()` 之前；它不负责更新参数，而是负责清理之前累积的梯度。

## 12. 一个需要特别注意的问题：src 没有参与训练

虽然 `make_batch()` 返回：

```python
src, tgt = make_batch(64, 6, device)
```

但后续模型实际接收的是：

```python
inp = tgt[:, :-1]
logits = model(inp)
```

因此模型并没有看到 `src`。

由于 `tgt` 是根据 `src` 反转后生成的，这段训练代码实际教模型的是：

**给定目标序列的正确前缀，预测这个目标序列的下一个 token。**

它并没有教模型根据 `src` 反转序列。

如果要完成真正的条件反转任务，模型必须能够利用源序列的信息。可以选择：

* **Encoder–Decoder Transformer**：Encoder 读取 `src`，Decoder 根据 Encoder 输出生成 `tgt`。
* **Decoder-only Transformer**：把源序列和目标前缀组织在同一个输入中，并设计合适的 Attention Mask 与 Loss Mask。

不能仅仅把当前的 `src` 生成代码保留下来，就认为模型已经学会了反转 `src`。

## 13. 每隔 200 步打印 Loss

```python
if step % 200 == 0:
    print(f'step {step:4d}, loss {loss.item():.4f}')
```

`step % 200 == 0` 表示每隔 200 个 step 打印一次训练信息。

因为训练步数是 `0` 到 `2000`，所以会打印 11 次，包括第 0 步和第 2000 步。

`loss.item()` 将只包含一个数值的 Tensor 转换为 Python 数字；`.4f` 表示保留四位小数。

例如：

```text
step    0, loss 3.20
step  200, loss 1.10
step  400, loss 0.45
...
```

这些数值只是格式示例，不是实际运行结果。

## 14. 返回模型与设备

```python
return model, device
```

训练函数结束后返回：

* `model`：经过训练的模型。
* `device`：模型使用的计算设备。

调用方式：

```python
model, device = train()
```

后续可以使用这个模型评估预测结果，或编写文本生成函数。

## 15. 完整训练代码

```python
def train():
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print('Using device:', device)

    torch.manual_seed(0)

    model = TinyTransformer(
        vocab_size=VOCAB,
        dim=64,
        depth=2,
        num_heads=4,
        max_len=16
    ).to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=3e-4
    )

    for step in range(2001):
        src, tgt = make_batch(64, 6, device)

        # Teacher Forcing：将目标序列错开一个位置
        inp = tgt[:, :-1]
        label = tgt[:, 1:]

        # 前向传播
        logits = model(inp)

        # 计算预测损失
        loss = F.cross_entropy(
            logits.reshape(-1, VOCAB),
            label.reshape(-1)
        )

        # 反向传播与参数更新
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # 打印训练进度
        if step % 200 == 0:
            print(
                f'step {step:4d}, '
                f'loss {loss.item():.4f}'
            )

    return model, device
```

## 16. 总结

训练一个神经网络的核心步骤是：

1. 准备训练数据。
2. 前向传播得到预测结果。
3. 计算预测结果与正确答案之间的 Loss。
4. 反向传播计算梯度。
5. 使用优化器更新参数。
6. 重复训练并观察 Loss 的变化。

对于当前代码，最关键的是区分两个概念：

* **Teacher Forcing**：让模型在训练时使用正确的目标前缀。
* **条件序列反转**：要求模型根据输入 `src` 生成它的反转结果。

当前代码实现了前者，但因为 `src` 没有输入模型，所以尚未真正实现后者。


## 5. 训练

In [17]:

def train():
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print('Using device:', device)

    torch.manual_seed(0)

    model = TinyTransformer(
        vocab_size=VOCAB,
        dim=64,
        depth=2,
        num_heads=4,
        max_len=16
    ).to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=3e-4
    )

    length = 6

    for step in range(2001):
        # 随机源序列
        # src, _ = make_batch(64, length, device)
        src, target = make_batch2(64, length, device)

        # 目标序列：BOS + 反转后的源序列
        bos = torch.full(
            (src.size(0), 1),
            BOS,
            dtype=torch.long,
            device=device
        )
        # target = torch.cat([bos, src.flip(1)], dim=1)
        # target = torch.cat([bos, src], dim=1)

        # 完整序列：[src, BOS, reversed_src]
        seq = torch.cat([src, target], dim=1)

        # Teacher Forcing：输入比完整序列少最后一个 token
        inp = seq[:, :-1]
        logits = model(inp)

        # 只训练目标部分，不训练源序列部分
        target_logits = logits[:, length:length + length, :]
        labels = seq[:, length + 1:]

        loss = F.cross_entropy(
            target_logits.reshape(-1, VOCAB),
            labels.reshape(-1)
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if step % 200 == 0:
            print(f'step {step:4d}, loss {loss.item():.4f}')

    return model, device

## 6. 推理（贪心解码）

下面保留了原始代码的推理逻辑，方便对照学习。注意：原代码的 `greedy_decode` 将 `src.flip(1)` 放进了输入前缀，这相当于把正确答案的一部分直接提供给模型；若要严格测试从原始序列生成反转序列，需要重新设计 decoder-only 的输入拼接方式。

In [12]:

def greedy_decode(model, src, max_new_tokens=6):
    model.eval()

    # 与训练格式保持一致：[原序列, BOS]
    bos = torch.full(
        (src.size(0), 1),
        BOS,
        dtype=torch.long,
        device=src.device
    )

    out = torch.cat([src, bos], dim=1)

    generated = []

    with torch.no_grad():
        for _ in range(max_new_tokens):
            logits = model(out)

            # 预测下一个 token
            next_tok = logits[:, -1, :].argmax(
                dim=-1, keepdim=True
            )

            generated.append(next_tok)
            out = torch.cat([out, next_tok], dim=1)

    return torch.cat(generated, dim=1)


def evaluate(model, device):
    src = torch.tensor(
        [[2, 3, 4, 5, 6, 7]],
        dtype=torch.long,
        device=device
    )

    pred = greedy_decode(model, src)

    print('input :', src.tolist())
    print('pred  :', pred.tolist())
    print('expect:', src.flip(1).tolist())

## 7. 运行训练与推理验证

In [18]:
model, device = train()
print('\n--- 推理验证 ---')
# evaluate(model, device)

Using device: cpu
step    0, loss 3.2673
step  200, loss 0.1028
step  400, loss 0.0239
step  600, loss 0.0121
step  800, loss 0.0075
step 1000, loss 0.0051
step 1200, loss 0.0037
step 1400, loss 0.0028
step 1600, loss 0.0022
step 1800, loss 0.0018
step 2000, loss 0.0014

--- 推理验证 ---


In [13]:
evaluate(model, device)

input : [[2, 3, 4, 5, 6, 7]]
pred  : [[7, 6, 5, 4, 3, 2]]
expect: [[7, 6, 5, 4, 3, 2]]


In [11]:
model.eval()

src = torch.tensor(
    [[2, 3, 4, 5, 6, 7]],
    dtype=torch.long,
    device=device
)

context = torch.cat([
    src,
    torch.full((1, 1), BOS, dtype=torch.long, device=device)
], dim=1)

pred = []

with torch.no_grad():
    for step in range(6):
        logits = model(context)
        next_token = logits[:, -1, :].argmax(dim=-1)

        pred.append(next_token.item())
        context = torch.cat(
            [context, next_token.unsqueeze(1)], dim=1
        )

print("input :", src[0].tolist())
print("pred  :", pred)
print("expect:", src[0].flip(0).tolist())

input : [2, 3, 4, 5, 6, 7]
pred  : [7, 6, 5, 4, 3, 2]
expect: [7, 6, 5, 4, 3, 2]


In [14]:
test_cases = [
    [2, 3, 4, 5, 6],
    [2, 3, 4, 5, 6, 7],
    [2, 3, 4, 5, 6, 7, 8],
]

for values in test_cases:
    src = torch.tensor([values], device=device)

    pred = greedy_decode(
        model,
        src,
        max_new_tokens=len(values)
    )

    expected = src.flip(1)

    print("input :", src[0].tolist())
    print("pred  :", pred[0].tolist())
    print("expect:", expected[0].tolist())
    print("correct:", torch.equal(pred, expected))
    print()

input : [2, 3, 4, 5, 6]
pred  : [14, 2, 6, 5, 4]
expect: [6, 5, 4, 3, 2]
correct: False

input : [2, 3, 4, 5, 6, 7]
pred  : [7, 6, 5, 4, 3, 2]
expect: [7, 6, 5, 4, 3, 2]
correct: True

input : [2, 3, 4, 5, 6, 7, 8]
pred  : [6, 5, 4, 3, 2, 5, 18]
expect: [8, 7, 6, 5, 4, 3, 2]
correct: False



In [23]:
model.eval()

src = torch.tensor(
    [[2, 3, 4, 5, 6, 7]],
    dtype=torch.long,
    device=device
)

# 从 [src, BOS] 开始生成
context = torch.cat([
    src,
    torch.full((1, 1), BOS, dtype=torch.long, device=device)
], dim=1)

pred = []

with torch.no_grad():
    for step in range(src.size(1)):
        logits = model(context)

        # 取最后一个位置预测的下一个 token
        next_token = logits[:, -1, :].argmax(dim=-1)

        pred.append(next_token.item())

        # 把预测结果追加到上下文
        context = torch.cat(
            [context, next_token.unsqueeze(1)], dim=1
        )

# 循环减 1：2 -> 21，3 -> 2，...，21 -> 20
expect = (2 + torch.remainder(src - 3, VOCAB - 2))[0].tolist()

print("input :", src[0].tolist())
print("pred  :", pred)
print("expect:", expect)

input : [2, 3, 4, 5, 6, 7]
pred  : [21, 2, 3, 4, 5, 6]
expect: [21, 2, 3, 4, 5, 6]
